In [1]:
# === CELDA 1: Definiciones del pipeline de limpieza ===
import os, re, glob
from bs4 import BeautifulSoup

def extract_filing_text(raw):
    """Extrae secciones <TEXT> de documentos principales (10-K, 10-Q, etc.)."""
    TARGET_TYPES = {"10-K", "10-Q", "10-K/A", "10-Q/A", "10-KSB", "10-QSB"}
    doc_pattern = re.compile(
        r'<DOCUMENT>\s*<TYPE>([^\n<]+)(.*?)</DOCUMENT>',
        re.DOTALL | re.IGNORECASE
    )
    parts = []
    for dtype, content in doc_pattern.findall(raw):
        if dtype.strip().upper() in TARGET_TYPES:
            # Extraer solo el contenido dentro de <TEXT>...</TEXT>
            text_match = re.search(r'<TEXT>(.*?)</TEXT>', content, re.DOTALL | re.IGNORECASE)
            if text_match:
                parts.append(text_match.group(1))
            else:
                parts.append(content)
    # Si no encontró documentos SGML, devolver todo (filing sin estructura SGML)
    return '\n'.join(parts) if parts else raw

def strip_binary(text):
    """Elimina líneas de contenido binario (UUEncoded, base64, etc.)."""
    lines = text.split('\n')
    clean = []
    for line in lines:
        stripped = line.strip()
        # Saltar líneas que parecen base64/uuencoded
        if len(stripped) > 60 and re.match(r'^[A-Za-z0-9+/=]{60,}$', stripped):
            continue
        if re.match(r'^M[A-Za-z0-9+/]{59,}$', stripped):  # UUEncoded
            continue
        # Saltar líneas begin/end de UUEncode
        if re.match(r'^begin \d{3} ', stripped):
            continue
        if stripped == 'end' and len(stripped) == 3:
            continue
        clean.append(line)
    return '\n'.join(clean)

def clean_html(text):
    """Convierte HTML a texto plano con BeautifulSoup."""
    if '<html' in text.lower() or '<table' in text.lower() or '<div' in text.lower():
        soup = BeautifulSoup(text, 'html.parser')
        # Eliminar scripts y estilos
        for tag in soup(['script', 'style', 'xbrl', 'ix:nonfraction',
                         'ix:nonnumeric', 'ix:header', 'ix:hidden']):
            tag.decompose()
        text = soup.get_text(separator=' ')
    return text

def clean_sgml_tags(text):
    """Elimina tags SGML residuales."""
    text = re.sub(r'<[A-Z][A-Z0-9/\s=".\-:]*>', ' ', text)
    return text

def strip_xbrl_refs(text):
    """Elimina líneas de referencias XBRL/FASB taxonomy."""
    lines = text.split('\n')
    clean = []
    for line in lines:
        if re.search(r'Topic \d+ -SubTopic', line):
            continue
        if re.search(r'us-gaap:|dei:|srt:', line, re.IGNORECASE):
            continue
        clean.append(line)
    return '\n'.join(clean)

def normalize_whitespace(text):
    """Colapsa espacios múltiples y líneas en blanco excesivas."""
    text = re.sub(r'[ \t]+', ' ', text)           # Múltiples espacios → uno
    text = re.sub(r'\n{3,}', '\n\n', text)         # Máx 2 saltos de línea
    text = re.sub(r'(\n )+', '\n', text)           # Líneas con solo espacio
    return text.strip()

def clean_filing(raw_text):
    """Pipeline completo de limpieza."""
    text = extract_filing_text(raw_text)
    text = strip_binary(text)
    text = clean_html(text)
    text = clean_sgml_tags(text)
    text = strip_xbrl_refs(text)
    text = normalize_whitespace(text)
    return text

print("✅ Funciones de limpieza definidas")


✅ Funciones de limpieza definidas


In [2]:
# === CELDA 2: Procesar todos los archivos ===
import time

INPUT_DIR   = "./sec_data/sec-edgar-filings"
OUTPUT_DIR  = "./sec_data/cleaned"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Encontrar todos los full-submission.txt
filings = []
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f == "full-submission.txt":
            filings.append(os.path.join(root, f))

print(f"📂 {len(filings)} archivos encontrados\n")

t0 = time.time()
results = []

for i, filepath in enumerate(filings):
    # Extraer ticker y tipo del path
    parts = filepath.replace("\\", "/").split("/")
    ticker = parts[-4] if len(parts) >= 4 else "UNKNOWN"
    filing_type = parts[-3] if len(parts) >= 3 else "UNKNOWN"
    date_folder = parts[-2] if len(parts) >= 2 else "UNKNOWN"

    with open(filepath, "r", encoding="utf-8", errors="ignore") as fh:
        raw = fh.read()

    original_size = len(raw)
    cleaned = clean_filing(raw)
    cleaned_size = len(cleaned)

    # Guardar
    out_name = f"{ticker}_{filing_type}_{date_folder}.txt"
    out_path = os.path.join(OUTPUT_DIR, out_name)
    with open(out_path, "w", encoding="utf-8") as fh:
        fh.write(cleaned)

    reduction = (1 - cleaned_size / original_size) * 100 if original_size > 0 else 0
    results.append({
        "ticker": ticker,
        "type": filing_type,
        "original_kb": original_size / 1024,
        "cleaned_kb": cleaned_size / 1024,
        "reduction": reduction
    })

    if (i + 1) % 25 == 0 or (i + 1) == len(filings):
        print(f"   [{i+1}/{len(filings)}] procesados...")

elapsed = time.time() - t0

print(f"\n{'=' * 60}")
print(f"✅ {len(results)} archivos limpiados en {elapsed:.1f}s")
print(f"📁 Guardados en: {OUTPUT_DIR}")

# Resumen
total_orig = sum(r['original_kb'] for r in results)
total_clean = sum(r['cleaned_kb'] for r in results)
print(f"\n📊 Tamaño total original:  {total_orig/1024:.1f} MB")
print(f"📊 Tamaño total limpiado:  {total_clean/1024:.1f} MB")
print(f"📊 Reducción promedio:     {sum(r['reduction'] for r in results)/len(results):.1f}%")


📂 150 archivos encontrados

   [25/150] procesados...
   [50/150] procesados...
   [75/150] procesados...
   [100/150] procesados...
   [125/150] procesados...
   [150/150] procesados...

✅ 150 archivos limpiados en 38.3s
📁 Guardados en: ./sec_data/cleaned

📊 Tamaño total original:  2017.4 MB
📊 Tamaño total limpiado:  0.0 MB
📊 Reducción promedio:     100.0%


In [3]:
# === CELDA 3: Verificación rápida de calidad ===
import random

cleaned_files = glob.glob(os.path.join(OUTPUT_DIR, "*.txt"))
sample = random.sample(cleaned_files, min(3, len(cleaned_files)))

for fpath in sample:
    fname = os.path.basename(fpath)
    with open(fpath, "r", encoding="utf-8") as fh:
        text = fh.read()

    lines = text.split('\n')
    narrative = [l for l in lines if len(l.strip()) > 80]

    # Verificaciones
    sgml_tags = re.findall(r'<[A-Z][A-Z0-9/]{2,}>', text)
    html_tags = re.findall(r'<(?:div|span|table|td|tr|p|html|body)\b', text, re.IGNORECASE)
    binary_lines = [l for l in lines if len(l.strip()) > 60
                    and re.match(r'^[A-Za-z0-9+/=]{60,}$', l.strip())]

    print(f"\n{'─' * 60}")
    print(f"📄 {fname}")
    print(f"   Tamaño: {len(text)/1024:.1f} KB")
    print(f"   Líneas totales: {len(lines)}")
    print(f"   Líneas narrativas (>80 chars): {len(narrative)}")
    print(f"   Tags SGML residuales: {len(sgml_tags)}")
    print(f"   Tags HTML residuales: {len(html_tags)}")
    print(f"   Líneas binarias residuales: {len(binary_lines)}")
    print(f"   Primeros 200 chars:")
    print(f"   \"{text[:200]}\"")

print(f"\n✅ Verificación completada")



────────────────────────────────────────────────────────────
📄 INTC_10-Q_0000050863-23-000069.txt
   Tamaño: 0.0 KB
   Líneas totales: 1
   Líneas narrativas (>80 chars): 0
   Tags SGML residuales: 0
   Tags HTML residuales: 0
   Líneas binarias residuales: 0
   Primeros 200 chars:
   ""

────────────────────────────────────────────────────────────
📄 GOOGL_10-Q_0001652044-23-000045.txt
   Tamaño: 0.0 KB
   Líneas totales: 1
   Líneas narrativas (>80 chars): 0
   Tags SGML residuales: 0
   Tags HTML residuales: 0
   Líneas binarias residuales: 0
   Primeros 200 chars:
   ""

────────────────────────────────────────────────────────────
📄 MSFT_10-K_0000950170-24-087843.txt
   Tamaño: 0.0 KB
   Líneas totales: 1
   Líneas narrativas (>80 chars): 0
   Tags SGML residuales: 0
   Tags HTML residuales: 0
   Líneas binarias residuales: 0
   Primeros 200 chars:
   ""

✅ Verificación completada


In [4]:
# === CELDA DIAGNÓSTICO: Examinar estructura de los archivos originales ===
import os, re

INPUT_DIR = "./sec_data/sec-edgar-filings"

# Tomar 3 archivos de ejemplo
samples = []
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f == "full-submission.txt":
            samples.append(os.path.join(root, f))
            if len(samples) >= 3:
                break
    if len(samples) >= 3:
        break

for fpath in samples:
    with open(fpath, "r", encoding="utf-8", errors="ignore") as fh:
        raw = fh.read()

    print(f"\n{'=' * 70}")
    print(f"📄 {fpath}")
    print(f"   Tamaño: {len(raw)/1024:.1f} KB")

    # 1. Buscar todos los <TYPE> en el archivo
    types_found = re.findall(r'<TYPE>([^\n<]+)', raw, re.IGNORECASE)
    print(f"\n   📋 TYPEs encontrados ({len(types_found)}):")
    for t in types_found:
        print(f"      - '{t.strip()}'")

    # 2. Verificar si hay bloques <DOCUMENT>
    doc_count = len(re.findall(r'<DOCUMENT>', raw, re.IGNORECASE))
    doc_end_count = len(re.findall(r'</DOCUMENT>', raw, re.IGNORECASE))
    print(f"\n   📦 <DOCUMENT> abiertos: {doc_count}")
    print(f"   📦 </DOCUMENT> cerrados: {doc_end_count}")

    # 3. Verificar si hay bloques <TEXT>
    text_count = len(re.findall(r'<TEXT>', raw, re.IGNORECASE))
    text_end_count = len(re.findall(r'</TEXT>', raw, re.IGNORECASE))
    print(f"\n   📝 <TEXT> abiertos: {text_count}")
    print(f"   📝 </TEXT> cerrados: {text_end_count}")

    # 4. Mostrar primeros 1000 chars para ver la estructura
    print(f"\n   🔍 Primeros 1000 caracteres:")
    print(f"   ---")
    for line in raw[:1000].split('\n'):
        print(f"   {line}")
    print(f"   ---")

print("\n✅ Diagnóstico completado")



📄 ./sec_data/sec-edgar-filings/AMZN/10-Q/0001018724-22-000019/full-submission.txt
   Tamaño: 8746.2 KB

   📋 TYPEs encontrados (72):
      - '10-Q'
      - 'EX-10.1'
      - 'EX-10.2'
      - 'EX-31.1'
      - 'EX-31.2'
      - 'EX-32.1'
      - 'EX-32.2'
      - 'EX-101.SCH'
      - 'EX-101.CAL'
      - 'EX-101.DEF'
      - 'EX-101.LAB'
      - 'EX-101.PRE'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 

In [5]:
# === CELDA DIAGNÓSTICO: Examinar estructura de los archivos originales ===
import os, re

INPUT_DIR = "./sec_data/sec-edgar-filings"

# Tomar 3 archivos de ejemplo
samples = []
for root, dirs, files in os.walk(INPUT_DIR):
    for f in files:
        if f == "full-submission.txt":
            samples.append(os.path.join(root, f))
            if len(samples) >= 3:
                break
    if len(samples) >= 3:
        break

for fpath in samples:
    with open(fpath, "r", encoding="utf-8", errors="ignore") as fh:
        raw = fh.read()

    print(f"\n{'=' * 70}")
    print(f"📄 {fpath}")
    print(f"   Tamaño: {len(raw)/1024:.1f} KB")

    # 1. Buscar todos los <TYPE> en el archivo
    types_found = re.findall(r'<TYPE>([^\n<]+)', raw, re.IGNORECASE)
    print(f"\n   📋 TYPEs encontrados ({len(types_found)}):")
    for t in types_found:
        print(f"      - '{t.strip()}'")

    # 2. Verificar si hay bloques <DOCUMENT>
    doc_count = len(re.findall(r'<DOCUMENT>', raw, re.IGNORECASE))
    doc_end_count = len(re.findall(r'</DOCUMENT>', raw, re.IGNORECASE))
    print(f"\n   📦 <DOCUMENT> abiertos: {doc_count}")
    print(f"   📦 </DOCUMENT> cerrados: {doc_end_count}")

    # 3. Verificar si hay bloques <TEXT>
    text_count = len(re.findall(r'<TEXT>', raw, re.IGNORECASE))
    text_end_count = len(re.findall(r'</TEXT>', raw, re.IGNORECASE))
    print(f"\n   📝 <TEXT> abiertos: {text_count}")
    print(f"   📝 </TEXT> cerrados: {text_end_count}")

    # 4. Mostrar primeros 1000 chars para ver la estructura
    print(f"\n   🔍 Primeros 1000 caracteres:")
    print(f"   ---")
    for line in raw[:1000].split('\n'):
        print(f"   {line}")
    print(f"   ---")

print("\n✅ Diagnóstico completado")



📄 ./sec_data/sec-edgar-filings/AMZN/10-Q/0001018724-22-000019/full-submission.txt
   Tamaño: 8746.2 KB

   📋 TYPEs encontrados (72):
      - '10-Q'
      - 'EX-10.1'
      - 'EX-10.2'
      - 'EX-31.1'
      - 'EX-31.2'
      - 'EX-32.1'
      - 'EX-32.2'
      - 'EX-101.SCH'
      - 'EX-101.CAL'
      - 'EX-101.DEF'
      - 'EX-101.LAB'
      - 'EX-101.PRE'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 'XML'
      - 

In [7]:
# Verificar que extract_filing_text captura el documento principal
with open("./sec_data/raw/AMZN/10-Q/full-submission.txt", "r", encoding="utf-8", errors="ignore") as f:
    raw = f.read()

extracted = extract_filing_text(raw)
print(f"Original: {len(raw):,} chars")
print(f"Extraído: {len(extracted):,} chars")
print(f"Reducción: {100*(1 - len(extracted)/len(raw)):.1f}%")
print(f"\nPrimeros 500 chars del texto extraído:\n{extracted[:500]}")


FileNotFoundError: [Errno 2] No such file or directory: './sec_data/raw/AMZN/10-Q/full-submission.txt'

In [8]:
import os

# Buscar la estructura real de sec_data
base = "./sec_data"
if os.path.exists(base):
    print("📂 Estructura de sec_data:\n")
    for root, dirs, files in os.walk(base):
        # Mostrar solo los primeros 3 niveles y archivos .txt
        depth = root.replace(base, "").count(os.sep)
        if depth < 4:
            indent = "  " * depth
            print(f"{indent}📁 {os.path.basename(root)}/")
            if depth >= 2:
                txt_files = [f for f in files if f.endswith(".txt")]
                if txt_files:
                    for tf in txt_files[:3]:
                        fpath = os.path.join(root, tf)
                        size_mb = os.path.getsize(fpath) / (1024*1024)
                        print(f"{indent}  📄 {tf} ({size_mb:.1f} MB)")
                    if len(txt_files) > 3:
                        print(f"{indent}  ... y {len(txt_files)-3} más")
else:
    # Buscar en otras ubicaciones comunes
    for alt in ["./data", "./raw", ".", "./sec_filings"]:
        if os.path.exists(alt):
            print(f"📁 Contenido de {alt}/:")
            for item in os.listdir(alt)[:20]:
                print(f"  {item}")
            print()


📂 Estructura de sec_data:

📁 sec_data/
  📁 sec-edgar-filings-redacted/
    📁 AMZN/
      📁 10-Q/
      📁 10-K/
    📁 AAPL/
      📁 10-Q/
      📁 10-K/
    📁 INTC/
      📁 10-Q/
      📁 10-K/
    📁 GOOGL/
      📁 10-Q/
      📁 10-K/
    📁 MSFT/
      📁 10-Q/
      📁 10-K/
    📁 META/
      📁 10-Q/
      📁 10-K/
  📁 cleaned/
  📁 sec-edgar-filings/
    📁 AMZN/
      📁 10-Q/
      📁 10-K/
    📁 AAPL/
      📁 10-Q/
      📁 10-K/
    📁 INTC/
      📁 10-Q/
      📁 10-K/
    📁 GOOGL/
      📁 10-Q/
      📁 10-K/
    📁 MSFT/
      📁 10-Q/
      📁 10-K/
    📁 META/
      📁 10-Q/
      📁 10-K/
  📁 sec-edgar-filings-redacted-v3/
    📁 AAPL/
      📁 10-K/
  📁 sec-edgar-filings-redacted-v2/
    📁 AAPL/
      📁 10-K/
  📁 redacted_v5/
    📁 AMZN/
      📁 10-Q/
      📁 10-K/
    📁 AAPL/
      📁 10-Q/
      📁 10-K/
    📁 INTC/
      📁 10-Q/
      📁 10-K/
    📁 GOOGL/
      📁 10-Q/
      📁 10-K/
    📁 MSFT/
      📁 10-Q/
      📁 10-K/
    📁 META/
      📁 10-Q/
      📁 10-K/


In [9]:
import os

# Explorar la estructura dentro de un ticker específico
base = "./sec_data/sec-edgar-filings/AMZN/10-Q"
print(f"📁 Contenido de {base}:\n")

for root, dirs, files in os.walk(base):
    depth = root.replace(base, "").count(os.sep)
    indent = "  " * depth
    print(f"{indent}📁 {os.path.basename(root)}/")
    for f in files[:5]:
        fpath = os.path.join(root, f)
        size_mb = os.path.getsize(fpath) / (1024*1024)
        print(f"{indent}  📄 {f} ({size_mb:.1f} MB)")
    if len(files) > 5:
        print(f"{indent}  ... y {len(files)-5} archivos más")


📁 Contenido de ./sec_data/sec-edgar-filings/AMZN/10-Q:

📁 10-Q/
  📁 0001018724-22-000019/
    📄 full-submission.txt (8.5 MB)
  📁 0001018724-23-000012/
    📄 full-submission.txt (7.8 MB)
  📁 0001018724-20-000021/
    📄 full-submission.txt (8.5 MB)
  📁 0001018724-20-000010/
    📄 full-submission.txt (6.5 MB)
  📁 0001018724-25-000036/
    📄 full-submission.txt (7.4 MB)
  📁 0001018724-23-000008/
    📄 full-submission.txt (7.5 MB)
  📁 0001018724-24-000130/
    📄 full-submission.txt (7.7 MB)
  📁 0001018724-22-000023/
    📄 full-submission.txt (8.3 MB)
  📁 0001018724-25-000123/
    📄 full-submission.txt (9.1 MB)
  📁 0001018724-22-000013/
    📄 full-submission.txt (8.3 MB)
  📁 0001018724-23-000018/
    📄 full-submission.txt (7.5 MB)
  📁 0001018724-19-000089/
    📄 full-submission.txt (7.5 MB)
  📁 0001018724-21-000020/
    📄 full-submission.txt (7.7 MB)
  📁 0001018724-21-000010/
    📄 full-submission.txt (6.7 MB)
  📁 0001018724-21-000028/
    📄 full-submission.txt (7.7 MB)
  📁 0001018724-20-000

In [10]:
import re, os

# 1. Leer un archivo de ejemplo
sample = "./sec_data/sec-edgar-filings/AMZN/10-Q/0001018724-20-000021/full-submission.txt"
with open(sample, "r", encoding="utf-8", errors="ignore") as f:
    raw = f.read()

print(f"Tamaño original: {len(raw):,} chars\n")

# 2. Ver todos los <TYPE> que hay en el archivo
types_found = re.findall(r"<TYPE>(.+)", raw)
print(f"Tipos de <DOCUMENT> encontrados ({len(types_found)}):")
for t in types_found:
    print(f"  • {t.strip()}")

# 3. Extraer SOLO el bloque <DOCUMENT> cuyo <TYPE> sea 10-Q (o 10-K)
TARGET_TYPES = {"10-K", "10-Q", "10-K/A", "10-Q/A"}
doc_pattern = re.compile(
    r"<DOCUMENT>\s*<TYPE>(?P<dtype>[^\n]+)\n(?P<body>.*?)</DOCUMENT>",
    re.DOTALL
)

extracted = ""
for m in doc_pattern.finditer(raw):
    dtype = m.group("dtype").strip()
    if dtype in TARGET_TYPES:
        body = m.group("body")
        # Extraer solo lo que está dentro de <TEXT>...</TEXT>
        text_match = re.search(r"<TEXT>(.*?)</TEXT>", body, re.DOTALL)
        if text_match:
            extracted = text_match.group(1)
        else:
            extracted = body
        print(f"\n✅ Bloque encontrado: <TYPE>{dtype}")
        print(f"   Tamaño del texto extraído: {len(extracted):,} chars")
        print(f"   Reducción: {100*(1 - len(extracted)/len(raw)):.1f}%")
        break

if not extracted:
    print("\n❌ No se encontró ningún bloque con TYPE en TARGET_TYPES")
else:
    print(f"\nPrimeros 500 chars del texto extraído:")
    print("-" * 60)
    print(extracted[:500])


Tamaño original: 8,937,901 chars

Tipos de <DOCUMENT> encontrados (66):
  • 10-Q
  • EX-31.1
  • EX-31.2
  • EX-32.1
  • EX-32.2
  • EX-101.SCH
  • EX-101.CAL
  • EX-101.DEF
  • EX-101.LAB
  • EX-101.PRE
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • XML
  • EXCEL
  • XML
  • XML
  • XML
  • JSON
  • ZIP

✅ Bloque encontrado: <TYPE>10-Q
   Tamaño del texto extraído: 2,188,808 chars
   Reducción: 75.5%

Primeros 500 chars del texto extraído:
------------------------------------------------------------

<XBRL>
<?xml version="1.0" ?><!--XBRL Document Created with Wdesk from Workiva--><!--Copyright 2020 Workiva--><!--r:36920fc5-0f6b-4d82-83f8-4152

In [11]:
from bs4 import BeautifulSoup
import re

# ── Funciones del pipeline ──────────────────────────────────────

def strip_binary(text):
    """Elimina líneas que parecen datos binarios/codificados"""
    lines = text.split("\n")
    clean = []
    for line in lines:
        stripped = line.strip()
        # Saltar líneas base64 o binarias (alta proporción de chars no-ASCII o muy largas sin espacios)
        if len(stripped) > 200 and " " not in stripped:
            continue
        if re.match(r'^[A-Za-z0-9+/=]{50,}$', stripped):
            continue
        clean.append(line)
    return "\n".join(clean)

def clean_html(text):
    """Convierte HTML a texto plano"""
    soup = BeautifulSoup(text, "html.parser")
    for tag in soup.find_all(["script", "style"]):
        tag.decompose()
    return soup.get_text(separator="\n")

def clean_sgml_tags(text):
    """Elimina tags SGML residuales"""
    return re.sub(r"<[^>]+>", "", text)

def strip_xbrl_refs(text):
    """Elimina líneas de referencias XBRL/FASB"""
    lines = text.split("\n")
    clean = []
    xbrl_patterns = re.compile(
        r"(Topic \d+|SubTopic|Section S\d+|Paragraph|ASC |"
        r"fasb\.org|xbrl\.org|xbrl\.sec\.gov|us-gaap:|dei:|"
        r"http://www\.|Member|Axis|LineItems|Abstract"
        r")", re.IGNORECASE
    )
    for line in lines:
        if xbrl_patterns.search(line):
            continue
        clean.append(line)
    return "\n".join(clean)

def normalize_whitespace(text):
    """Normaliza espacios y líneas en blanco"""
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

# ── Aplicar pipeline paso a paso ────────────────────────────────

steps = {
    "1. Texto extraído (raw)": extracted,
    "2. strip_binary":         None,
    "3. clean_html":           None,
    "4. clean_sgml_tags":      None,
    "5. strip_xbrl_refs":      None,
    "6. normalize_whitespace": None,
}

text = extracted
for i, (name, _) in enumerate(steps.items()):
    if i == 0:
        steps[name] = text
        continue
    func = [strip_binary, clean_html, clean_sgml_tags, strip_xbrl_refs, normalize_whitespace][i-1]
    text = func(text)
    steps[name] = text

print("=" * 65)
print(f"{'Paso':<35} {'Chars':>10}  {'Líneas':>8}")
print("=" * 65)
for name, t in steps.items():
    print(f"{name:<35} {len(t):>10,}  {t.count(chr(10)):>8,}")
print("=" * 65)

# ── Muestra del resultado final ─────────────────────────────────
final = text
narrative = [l.strip() for l in final.split("\n") if len(l.strip()) > 80]
print(f"\nLíneas narrativas (>80 chars): {len(narrative)}")
print(f"\nPrimeras 5 líneas narrativas:")
print("-" * 65)
for line in narrative[:5]:
    print(line[:120])
    print()

print(f"\nÚltimos 1000 chars del texto final:")
print("-" * 65)
print(final[-1000:])


Paso                                     Chars    Líneas
1. Texto extraído (raw)              2,188,808         7
2. strip_binary                      2,188,808         7
3. clean_html                          183,686     5,725
4. clean_sgml_tags                     183,686     5,725
5. strip_xbrl_refs                     171,495     5,452
6. normalize_whitespace                171,476     5,442

Líneas narrativas (>80 chars): 319

Primeras 5 líneas narrativas:
-----------------------------------------------------------------
QUARTERLY REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

TRANSITION REPORT PURSUANT TO SECTION 13 OR 15(d) OF THE SECURITIES EXCHANGE ACT OF 1934

(Address and telephone number, including area code, of registrant’s principal executive offices)

Indicate by check mark whether the registrant (1) has filed all reports required to be filed by Section 13 or 15(d) of t

Indicate by check mark whether the registrant has submitted electroni

In [13]:
import re, os, glob
from bs4 import BeautifulSoup

# ── Funciones del pipeline v6 (final) ──────────────────────────

def extract_filing_text(raw):
    TARGET_TYPES = {"10-K", "10-Q", "10-K/A", "10-Q/A"}
    doc_pattern = re.compile(
        r"<DOCUMENT>\s*<TYPE>(?P<dtype>[^\n]+)\n(?P<body>.*?)</DOCUMENT>",
        re.DOTALL
    )
    for m in doc_pattern.finditer(raw):
        dtype = m.group("dtype").strip()
        if dtype in TARGET_TYPES:
            body = m.group("body")
            text_match = re.search(r"<TEXT>(.*?)</TEXT>", body, re.DOTALL)
            return text_match.group(1) if text_match else body
    return raw  # fallback: devolver todo si no hay match

def strip_binary(text):
    lines = text.split("\n")
    clean = []
    for line in lines:
        stripped = line.strip()
        if len(stripped) > 200 and " " not in stripped:
            continue
        if re.match(r'^[A-Za-z0-9+/=]{50,}$', stripped):
            continue
        clean.append(line)
    return "\n".join(clean)

def clean_html(text):
    soup = BeautifulSoup(text, "html.parser")
    for tag in soup.find_all(["script", "style"]):
        tag.decompose()
    return soup.get_text(separator="\n")

def clean_sgml_tags(text):
    return re.sub(r"<[^>]+>", "", text)

def strip_xbrl_refs(text):
    xbrl_patterns = re.compile(
        r"(Topic \d+|SubTopic|Section S\d+|ASC \d|"
        r"fasb\.org|xbrl\.org|xbrl\.sec\.gov|us-gaap:|dei:|"
        r"Member$|Axis$|LineItems$|Abstract$"
        r")", re.IGNORECASE
    )
    lines = text.split("\n")
    return "\n".join(l for l in lines if not xbrl_patterns.search(l))

def fix_xbrl_fragmentation(text):
    """Repara texto fragmentado por Inline XBRL (ej: 'A MAZON . COM' → 'AMAZON.COM')"""
    # Unir letras sueltas separadas por espacios: "A M A Z O N" → "AMAZON"
    text = re.sub(r'\b([A-Z]) (?=[A-Z]\b)', r'\1', text)
    # Eliminar espacios antes de puntuación
    text = re.sub(r'\s+([.,;:!?)])', r'\1', text)
    # Eliminar espacios después de paréntesis abierto
    text = re.sub(r'(\()\s+', r'\1', text)
    return text

def normalize_whitespace(text):
    text = re.sub(r"[ \t]+", " ", text)
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()

def clean_filing(raw):
    text = extract_filing_text(raw)
    text = strip_binary(text)
    text = clean_html(text)
    text = clean_sgml_tags(text)
    text = strip_xbrl_refs(text)
    text = fix_xbrl_fragmentation(text)
    text = normalize_whitespace(text)
    return text

# ── Batch processing ────────────────────────────────────────────

input_base = "./sec_data/sec-edgar-filings"
output_base = "./sec_data/cleaned"
os.makedirs(output_base, exist_ok=True)

pattern = os.path.join(input_base, "*", "*", "*", "full-submission.txt")
files = sorted(glob.glob(pattern))
print(f"Archivos encontrados: {len(files)}\n")

stats = []
for i, fpath in enumerate(files):
    parts = fpath.replace("\\", "/").split("/")
    # .../sec-edgar-filings/TICKER/FILING_TYPE/ACCESSION/full-submission.txt
    ticker = parts[-4]
    filing_type = parts[-3]
    accession = parts[-2]

    with open(fpath, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()

    cleaned = clean_filing(raw)

    # Guardar
    out_dir = os.path.join(output_base, ticker, filing_type)
    os.makedirs(out_dir, exist_ok=True)
    out_file = os.path.join(out_dir, f"{accession}.txt")
    with open(out_file, "w", encoding="utf-8") as f:
        f.write(cleaned)

    narrative_lines = sum(1 for l in cleaned.split("\n") if len(l.strip()) > 80)
    stats.append({
        "ticker": ticker,
        "type": filing_type,
        "accession": accession,
        "raw_chars": len(raw),
        "clean_chars": len(cleaned),
        "reduction_pct": round(100 * (1 - len(cleaned)/len(raw)), 1),
        "narrative_lines": narrative_lines
    })

    if (i + 1) % 25 == 0 or i == 0 or i == len(files) - 1:
        print(f"  [{i+1:3d}/{len(files)}] {ticker}/{filing_type}/{accession} → "
              f"{len(cleaned):>8,} chars ({stats[-1]['reduction_pct']}% red.) "
              f"| {narrative_lines} líneas narrativas")

# ── Resumen ─────────────────────────────────────────────────────
import statistics
clean_sizes = [s["clean_chars"] for s in stats]
narr_counts = [s["narrative_lines"] for s in stats]
reductions = [s["reduction_pct"] for s in stats]

print(f"\n{'='*65}")
print(f"✅ {len(stats)} archivos procesados")
print(f"{'='*65}")
print(f"  Chars limpios  → avg: {statistics.mean(clean_sizes):,.0f} | "
      f"min: {min(clean_sizes):,} | max: {max(clean_sizes):,}")
print(f"  Reducción      → avg: {statistics.mean(reductions):.1f}%")
print(f"  Líneas narrat. → avg: {statistics.mean(narr_counts):.0f} | "
      f"min: {min(narr_counts)} | max: {max(narr_counts)}")

# Resumen por ticker
tickers = sorted(set(s["ticker"] for s in stats))
print(f"\n{'Ticker':<8} {'Archivos':>8} {'Avg chars':>12} {'Avg narr.':>10}")
print("-" * 42)
for t in tickers:
    t_stats = [s for s in stats if s["ticker"] == t]
    avg_c = statistics.mean([s["clean_chars"] for s in t_stats])
    avg_n = statistics.mean([s["narrative_lines"] for s in t_stats])
    print(f"{t:<8} {len(t_stats):>8} {avg_c:>12,.0f} {avg_n:>10.0f}")



Archivos encontrados: 150

  [  1/150] AAPL/10-K/0000320193-21-000105 →  231,216 chars (97.8% red.) | 492 líneas narrativas
  [ 25/150] AAPL/10-Q/0000320193-26-000013 →   89,921 chars (98.5% red.) | 149 líneas narrativas
  [ 50/150] AMZN/10-Q/0001018724-26-000014 →  187,045 chars (97.8% red.) | 364 líneas narrativas
  [ 75/150] GOOGL/10-Q/0001652044-26-000071 →  215,515 chars (98.6% red.) | 468 líneas narrativas
  [100/150] INTC/10-Q/0000050863-26-000157 →  169,254 chars (98.3% red.) | 333 líneas narrativas
  [125/150] META/10-Q/0001628280-26-028526 →  393,138 chars (95.9% red.) | 633 líneas narrativas
  [150/150] MSFT/10-Q/0001564590-23-000733 →  268,077 chars (98.8% red.) | 463 líneas narrativas

✅ 150 archivos procesados
  Chars limpios  → avg: 230,587 | min: 60,487 | max: 535,835
  Reducción      → avg: 98.2%
  Líneas narrat. → avg: 430 | min: 115 | max: 1019

Ticker   Archivos    Avg chars  Avg narr.
------------------------------------------
AAPL           25      109,853        

In [14]:
import json, os, glob, re

# ── Parámetros ──────────────────────────────────────────────────
CHUNK_SIZE = 4000       # ~4000 chars por chunk
CHUNK_OVERLAP = 800     # ~800 chars de solapamiento
CHUNKS_OUTPUT = "./sec_data/chunks/all_chunks.jsonl"
os.makedirs(os.path.dirname(CHUNKS_OUTPUT), exist_ok=True)

# ── Función de chunking ────────────────────────────────────────
def chunk_text(text, size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    chunks = []
    start = 0
    while start < len(text):
        end = start + size
        chunk = text[start:end]
        # Intentar cortar en un salto de línea para no partir oraciones
        if end < len(text):
            last_newline = chunk.rfind("\n")
            if last_newline > size * 0.5:  # solo si el corte no es demasiado agresivo
                end = start + last_newline + 1
                chunk = text[start:end]
        chunks.append(chunk.strip())
        start = end - overlap
    return [c for c in chunks if len(c) > 50]  # filtrar chunks vacíos

# ── Procesar archivos limpios ───────────────────────────────────
cleaned_base = "./sec_data/cleaned"
pattern = os.path.join(cleaned_base, "*", "*", "*.txt")
files = sorted(glob.glob(pattern))
print(f"Archivos limpios encontrados: {len(files)}\n")

all_chunks = []
for fpath in files:
    parts = fpath.replace("\\", "/").split("/")
    ticker = parts[-3]
    filing_type = parts[-2]
    accession = parts[-1].replace(".txt", "")

    with open(fpath, "r", encoding="utf-8") as f:
        text = f.read()

    chunks = chunk_text(text)
    for idx, chunk in enumerate(chunks):
        all_chunks.append({
            "id": f"{ticker}_{filing_type}_{accession}_{idx:04d}",
            "ticker": ticker,
            "filing_type": filing_type,
            "accession": accession,
            "chunk_index": idx,
            "text": chunk,
            "char_count": len(chunk)
        })

# ── Guardar JSONL ───────────────────────────────────────────────
with open(CHUNKS_OUTPUT, "w", encoding="utf-8") as f:
    for chunk in all_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + "\n")

# ── Estadísticas ────────────────────────────────────────────────
import statistics
char_counts = [c["char_count"] for c in all_chunks]
chunks_per_ticker = {}
for c in all_chunks:
    chunks_per_ticker.setdefault(c["ticker"], []).append(c)

print(f"{'='*60}")
print(f"✅ {len(all_chunks):,} chunks generados → {CHUNKS_OUTPUT}")
print(f"{'='*60}")
print(f"  Chars por chunk → avg: {statistics.mean(char_counts):,.0f} | "
      f"min: {min(char_counts):,} | max: {max(char_counts):,}")
print(f"  Archivo: {os.path.getsize(CHUNKS_OUTPUT)/1024/1024:.1f} MB")
print(f"\n{'Ticker':<8} {'Chunks':>8} {'Avg chars':>12}")
print("-" * 32)
for t in sorted(chunks_per_ticker):
    t_chunks = chunks_per_ticker[t]
    avg_c = statistics.mean([c["char_count"] for c in t_chunks])
    print(f"{t:<8} {len(t_chunks):>8} {avg_c:>12,.0f}")


Archivos limpios encontrados: 150

✅ 11,940 chunks generados → ./sec_data/chunks/all_chunks.jsonl
  Chars por chunk → avg: 3,682 | min: 60 | max: 4,000
  Archivo: 46.5 MB

Ticker     Chunks    Avg chars
--------------------------------
AAPL          936        3,708
AMZN         1704        3,670
GOOGL        1739        3,781
INTC         1872        3,696
META         3307        3,591
MSFT         2382        3,724


In [17]:
# Instalar DVC (una sola vez)
pip install dvc dvc-gdrive  # o dvc-s3, dvc-gcs según tu remote

# Inicializar DVC en tu repo
cd tu-proyecto
dvc init

# Trackear los datos con DVC (no con git)
dvc add sec_data/cleaned
dvc add sec_data/chunks/all_chunks.jsonl

# Esto genera archivos .dvc — esos sí van a git
git add sec_data/cleaned.dvc sec_data/chunks/all_chunks.jsonl.dvc .dvc/.gitignore
git commit -m "feat: track cleaned corpus and chunks with DVC v1"


SyntaxError: invalid syntax (2431821076.py, line 2)